# S2 — 缺失值不是刪掉就好：Missing Data Strategy（Solution）

> **時間**：2 小時（講授 70min + 課堂練習 40min + QA 10min）  
> **資料**：`datasets/titanic/train.csv`（講授）、`datasets/house-prices/train.csv`（練習）  
> **學完能幹嘛**：判斷缺值屬於哪種機制，選擇最合適的填補策略，而不是無腦 `dropna()`

---

## 為什麼缺值處理這麼重要？

**一句話口訣：缺值有三種死法：隨機缺 (MCAR)、有規律缺 (MAR)、故意缺 (MNAR) — 搞錯了，你的分析就廢了。**

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

df = pd.read_csv('datasets/titanic/train.csv')
print(f'Titanic: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'缺值欄位：')
print(df.isnull().sum()[df.isnull().sum() > 0])

---
## 核心觀念 1／3 — NA 全景圖

In [ ]:
na_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
na_pct = na_pct[na_pct > 0]

fig, ax = plt.subplots(figsize=(8, 4))
na_pct.plot(kind='bar', color='coral', edgecolor='white', ax=ax)
ax.set_title('缺值比例 (%)', fontsize=14)
ax.set_ylabel('%')
for i, v in enumerate(na_pct):
    ax.text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, ax=ax)
ax.set_title('缺值分布熱力圖', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
na_cols = df.columns[df.isnull().any()]
na_corr = df[na_cols].isnull().corr()

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(na_corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('缺值相關性矩陣', fontsize=14)
plt.tight_layout()
plt.show()

---
## 核心觀念 2／3 — 三種缺失機制

In [ ]:
# Embarked — MCAR
print('=== Embarked (MCAR) ===')
print(df[df['Embarked'].isnull()][['Name', 'Pclass', 'Fare', 'Survived']])

In [ ]:
# Age — MAR
print('=== Age (MAR) ===')
age_na_by_class = df.groupby('Pclass')['Age'].apply(lambda x: x.isnull().mean() * 100)
print(age_na_by_class.round(1))

In [ ]:
# Cabin — MNAR
print('=== Cabin (MNAR) ===')
df['has_cabin'] = df['Cabin'].notnull().astype(int)
print(df.groupby('has_cabin')['Survived'].mean().round(3))

fig, ax = plt.subplots(figsize=(6, 4))
df.groupby('has_cabin')['Survived'].mean().plot(
    kind='bar', color=['#ff9999', '#66b2ff'], edgecolor='white', ax=ax
)
ax.set_title('有/沒有 Cabin 的存活率', fontsize=14)
ax.set_xticklabels(['No Cabin', 'Has Cabin'], rotation=0)
plt.tight_layout()
plt.show()

---
## 核心觀念 3／3 — 填補策略工具箱

In [ ]:
df_clean = df.copy()

# Embarked → 眾數
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])

# Age → 分組中位數
df_clean['Age'] = df_clean.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.median()))

# Cabin → has_cabin 旗標
df_clean['has_cabin'] = df_clean['Cabin'].notnull().astype(int)

# 驗證
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.kdeplot(df['Age'].dropna(), fill=True, ax=axes[0], color='steelblue')
axes[0].set_title('填補前 Age 分布')
sns.kdeplot(df_clean['Age'], fill=True, ax=axes[1], color='coral')
axes[1].set_title('填補後 Age 分布')
plt.tight_layout()
plt.show()

---
## 課堂練習 — Solution

### 🟢 送分題

In [ ]:
# 🟢 送分題 Solution
hp = pd.read_csv('datasets/house-prices/train.csv')

# 1-2. 前 15 個 NA% bar chart
hp_na = (hp.isnull().mean() * 100).sort_values(ascending=False)
hp_na_top15 = hp_na[hp_na > 0].head(15)

fig, ax = plt.subplots(figsize=(10, 5))
hp_na_top15.plot(kind='bar', color='coral', edgecolor='white', ax=ax)
ax.set_title('House Prices — 缺值比例 Top 15', fontsize=14)
ax.set_ylabel('%')
for i, v in enumerate(hp_na_top15):
    ax.text(i, v + 0.5, f'{v:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

# 3. 缺值超過 50% 的欄位
over_50 = hp_na[hp_na > 50]
print(f'\n缺值超過 50% 的欄位有 {len(over_50)} 個：')
print(over_50)

### 🟡 核心題

In [ ]:
# 🟡 核心題 Solution

# 1. PoolQC 的缺失機制
print('=== PoolQC 分析 ===')
print(f'PoolQC 缺值數：{hp["PoolQC"].isnull().sum()} / {len(hp)}')

# 2. PoolQC 缺值時，PoolArea 是多少？
pool_area_when_na = hp[hp['PoolQC'].isnull()]['PoolArea']
print(f'\nPoolQC 缺值時的 PoolArea：')
print(pool_area_when_na.describe())

pool_area_when_exists = hp[hp['PoolQC'].notnull()]['PoolArea']
print(f'\nPoolQC 有值時的 PoolArea：')
print(pool_area_when_exists.describe())

# 3. 結論
print('\n→ PoolQC 缺值的房子 PoolArea 全部是 0 — 沒有泳池就沒有泳池品質')
print('→ 這是典型的 MNAR：缺失本身就是資訊（= 沒有泳池）')
print('→ 策略：填 "None" 或建立 has_pool 二元特徵')

### 🔴 挑戰題

In [ ]:
# 🔴 挑戰題 Solution

# 1. 按 Neighborhood 分組看 LotFrontage 缺值率
lot_na_by_nbhd = hp.groupby('Neighborhood')['LotFrontage'].apply(
    lambda x: x.isnull().mean() * 100
).sort_values(ascending=False)

print('各社區的 LotFrontage 缺值率：')
print(lot_na_by_nbhd.round(1).head(10))
print(f'\n缺值率範圍：{lot_na_by_nbhd.min():.1f}% ~ {lot_na_by_nbhd.max():.1f}%')
print('→ 各社區缺值率差異不大，偏向 MCAR/MAR')
print('→ 但不同社區的 LotFrontage 中位數不同，用分組填補更合理')

In [ ]:
# 2. 分組填補
hp_clean = hp.copy()
hp_clean['LotFrontage'] = hp_clean.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)
print(f'填補後 LotFrontage 缺值：{hp_clean["LotFrontage"].isnull().sum()}')

# 3. 前後 KDE 比較
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.kdeplot(hp['LotFrontage'].dropna(), fill=True, ax=axes[0], color='steelblue')
axes[0].set_title('填補前 LotFrontage 分布', fontsize=12)

sns.kdeplot(hp_clean['LotFrontage'], fill=True, ax=axes[1], color='coral')
axes[1].set_title('填補後 LotFrontage 分布', fontsize=12)

plt.tight_layout()
plt.show()
print('→ 分布形狀基本保持，分組填補成功')

---
## 小結與速查表

| 想做什麼 | 怎麼寫 |
|:---------|:-------|
| 缺值計數 | `df.isnull().sum()` |
| 缺值比例 | `df.isnull().mean() * 100` |
| 缺值熱力圖 | `sns.heatmap(df.isnull(), cbar=False)` |
| 缺值相關性 | `df[na_cols].isnull().corr()` |
| 刪除高缺值列 | `df.dropna(thresh=N)` |
| 全域填補 | `df['col'].fillna(df['col'].median())` |
| 分組填補 | `df.groupby('g')['col'].transform(lambda x: x.fillna(x.median()))` |
| 缺值旗標 | `df['col_missing'] = df['col'].isnull().astype(int)` |

**下節預告 S3**：缺值處理完了，接下來要深入每個欄位的分布 → **Distribution & Outlier Analysis**